# DeepGuard — Large DF40 Benchmark

This notebook runs a large, reproducible DF40 evaluation from Google Drive. It does not install the obsolete `lir` package and does not install the old DF40 `install.sh`, which pins Python-era packages incompatible with modern Colab. The official DF40 processed test set is ~93 GB and is supplied separately by the DF40 project.


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, os, json
drive.mount('/content/drive', force_remount=False)
ROOT=Path('/content/drive/MyDrive/DeepGuard')
(ROOT/'benchmark/df40').mkdir(parents=True,exist_ok=True)
print(ROOT)
print(subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))


In [ ]:
%cd /content
if not Path('/content/deepguard-forensic-lr').exists():
    !git clone https://github.com/geradts/deepguard-forensic-lr.git
if not Path('/content/DeepfakeBench_DF40').exists():
    !git clone https://github.com/YZY-stack/DF40.git /content/DeepfakeBench_DF40
print('repositories ready')


## Drive layout

Put the official processed DF40 test data and dataset JSON manifests on Drive. Recommended layout:

`My Drive/DeepGuard/datasets/DF40` — processed DF40 test data

`My Drive/DeepGuard/models/df40/xception.pth` — official DF40 Xception checkpoint

The DF40 repository documents the Google Drive download locations for the processed test set, JSON files and checkpoints.


In [ ]:
DF40_DATA=ROOT/'datasets/DF40'
WEIGHTS=ROOT/'models/df40/xception.pth'
JSON_DIR=Path('/content/DeepfakeBench_DF40/preprocessing/dataset_json')
print('DF40 data exists:', DF40_DATA.exists())
print('Weights exist:', WEIGHTS.exists())
print('JSON dir exists:', JSON_DIR.exists())
if JSON_DIR.exists(): print('JSON count:',len(list(JSON_DIR.glob('*.json'))))


In [ ]:
# Copy/link the Drive dataset into the location expected by the DF40 code without duplicating files.
DATA_LINK=Path('/content/DeepfakeBench_DF40/datasets/DF40')
if not DATA_LINK.exists(): DATA_LINK.symlink_to(DF40_DATA, target_is_directory=True)
print('Dataset link:', DATA_LINK, '->', DATA_LINK.resolve())


In [ ]:
# Important: do NOT run DF40/install.sh in modern Colab. It pins obsolete NumPy/PyTorch versions.
# First verify that the current Colab PyTorch can import the DF40 code.
import torch
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())


In [ ]:
# Dry-run protocol 3: known-domain to unknown-forgery/domain test set.
%cd /content/deepguard-forensic-lr
!python scripts/run_df40_large_benchmark.py \
  --drive-root '/content/drive/MyDrive/DeepGuard' \
  --df40-root '/content/DeepfakeBench_DF40' \
  --weights '/content/drive/MyDrive/DeepGuard/models/df40/xception.pth' \
  --protocol p3 --detector xception --dry-run


## First real run

Run this only after the dry-run reports no missing files. The result is stored under `DeepGuard/benchmark/df40/xception/p3/`.


In [ ]:
!python scripts/run_df40_large_benchmark.py \
  --drive-root '/content/drive/MyDrive/DeepGuard' \
  --df40-root '/content/DeepfakeBench_DF40' \
  --weights '/content/drive/MyDrive/DeepGuard/models/df40/xception.pth' \
  --protocol p3 --detector xception


In [ ]:
# Protocol 2: same forgery types, different data domain.
!python scripts/run_df40_large_benchmark.py \
  --drive-root '/content/drive/MyDrive/DeepGuard' \
  --df40-root '/content/DeepfakeBench_DF40' \
  --weights '/content/drive/MyDrive/DeepGuard/models/df40/xception.pth' \
  --protocol p2 --detector xception


## Next stage: FTCN + DeepGuard-LR

After the Xception run is successful, we add FTCN scores and the independent DeepGuard-LR fusion. We do not calibrate on the external/open-set test data.
